# L04 — Structured Streaming Fundamentals

**Exam Section:** Section 1 — Developing Code for Data Processing using Python and SQL  
**Exam Objective:** Apply Structured Streaming stateful-processing semantics, including watermarks, output modes, foreachBatch, and checkpoints for fault-tolerant exactly-once state recovery.

## Learning Objectives
- **Use** `readStream` and `writeStream` to build a Structured Streaming query in PySpark.
- **Explain** the difference between `append`, `update`, and `complete` output modes and which aggregations each allows.
- **Choose** the appropriate trigger (`processingTime` vs `availableNow`) for a given use case.
- **Implement** watermarking to bound state and handle late-arriving events in windowed aggregations.
- **Use** `foreachBatch` with a checkpoint location to achieve exactly-once delivery into a Delta table via MERGE.

> **Note:** This lesson uses both **PySpark** and **SQL**. Streaming-specific APIs (`readStream`, `writeStream`, triggers, watermarks) are PySpark only in plain notebooks. SQL equivalents exist inside Lakeflow Declarative Pipelines (formerly DLT) using `FROM STREAM(...)`.  
> **Compute:** Serverless compute is preferred. All demos run on serverless unless noted.

In [0]:
%python
# ═══════════════════════════════════════════════════════════════════
# CLASSROOM SETUP — Run this cell first
# Creates schema, volume, and ~1,000 rows of synthetic healthcare data
# All data is fake — no real PHI (Protected Health Information)
# ═══════════════════════════════════════════════════════════════════

from datetime import datetime, timedelta  # Python datetime for event_time generation
import random                               # Python random for reproducible fake data

# --- Configuration constants (reused by ALL subsequent cells) ---
CATALOG = "main"                                        # Use 'main' catalog (the default in this workspace)
SCHEMA = "dbacademy_streaming"                           # Per-lesson schema name
FULL_SCHEMA = f"{CATALOG}.{SCHEMA}"                     # Fully qualified schema name
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/data"            # UC volume path for file storage

# --- Drop and recreate schema (makes setup idempotent — safe to re-run) ---
spark.sql(f"DROP SCHEMA IF EXISTS {FULL_SCHEMA} CASCADE")  # Drop everything if schema exists
spark.sql(f"CREATE SCHEMA {FULL_SCHEMA}")                   # Create fresh schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FULL_SCHEMA}.data")  # Create volume for file storage

# --- Fixed seed so data is identical every run ---
random.seed(42)

# --- Generate patients (50 rows) ---
patient_ids = list(range(1001, 1051))   # 50 patient IDs
patients = [
    (pid, f"Patient-{pid}", random.choice(["M", "F"]), random.randint(18, 90))
    for pid in patient_ids
]
spark.createDataFrame(patients, "patient_id INT, name STRING, gender STRING, age INT") \
    .write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.patients")

# --- Generate encounters (200 rows) ---
encounters = [
    (i, random.choice(patient_ids),
     f"2026-10-{random.randint(1, 14):02d}",
     random.choice(["ER", "Inpatient", "Outpatient"]))
    for i in range(1, 201)
]
spark.createDataFrame(encounters, "encounter_id INT, patient_id INT, encounter_date STRING, encounter_type STRING") \
    .write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.encounters")

# --- Generate claims (100 rows) ---
claims = [
    (i, random.randint(1, 200),
     round(random.uniform(100, 50000), 2),
     random.choice(["approved", "denied", "pending"]))
    for i in range(1, 101)
]
spark.createDataFrame(claims, "claim_id INT, encounter_id INT, amount DOUBLE, status STRING") \
    .write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.claims")

# --- Generate lab results (150 rows) ---
lab_results = [
    (i, random.choice(patient_ids),
     random.choice(["glucose", "cholesterol", "hemoglobin", "sodium"]),
     round(random.uniform(50, 200), 1),
     f"2026-10-{random.randint(1, 14):02d} {random.randint(0, 23):02d}:00:00")
    for i in range(1, 151)
]
spark.createDataFrame(lab_results, "lab_id INT, patient_id INT, test_name STRING, result_value DOUBLE, result_time STRING") \
    .write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.lab_results")

# --- Write FHIR-like JSON to the UC volume (20 resources) ---
# FHIR = Fast Healthcare Interoperability Resources — a common healthcare data standard
fhir_df = spark.range(20).selectExpr(
    "id AS resource_id",
    """to_json(named_struct(
        'resourceType', 'Patient',
        'id', cast(id % 50 AS string),
        'name', array(named_struct('family', concat('Family-', cast(id AS string)))),
        'gender', CASE WHEN id % 2 = 0 THEN 'male' ELSE 'female' END
    )) AS fhir_json"""
)
fhir_df.write.mode("overwrite").json(f"{VOLUME}/fhir_resources")  # Store as JSON in the volume

# --- Generate vital_sign_events: THE STREAMING SOURCE (1,020 rows) ---
# This Delta table simulates a stream of patient vital-sign readings.
# It includes LATE events, EXACT COPIES (same event_id + event_time),
# and RE-SENT events (same event_id, DIFFERENT event_time 1-5 min later).
# 5 additional re-sent events (event_time ~15h later) are stored separately
# in vital_sign_resend and inserted during Demo 7.
base_dt = datetime(2024, 9, 22, 0, 0, 0)  # Fixed reference time for event_time values

# Position sets for special events (no overlap between sets)
late_positions      = {50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
                       550, 600, 650, 700, 750, 800, 850, 900, 950, 1000}    # 20 late
copy_positions       = {20, 40, 60, 80, 120, 140, 160, 180, 220, 240}        # 10 exact copies
within_wm_positions  = {30, 70, 110, 130, 170, 190, 230, 270, 290, 310}       # 10 re-sent (within watermark)
outside_wm_positions = {5, 15, 25, 35, 45}                                     # 5 re-sent (outside watermark)

events = []           # Main streaming source (1,020 rows)
outside_resends = []  # Re-sent events inserted later in Demo 7 (5 rows)

for i in range(1, 1001):
    if i in late_positions:
        minutes_offset = i - 30   # Late: event_time 30 min behind expected
    else:
        minutes_offset = i        # Normal: event_time tracks insertion order
    event_time = base_dt + timedelta(minutes=minutes_offset)
    row = (
        i,                                          # event_id (unique per original event)
        random.choice(patient_ids),                  # patient_id
        event_time,                                 # event_time
        random.randint(50, 120),                     # heart_rate (bpm)
        random.randint(80, 180),                    # systolic_bp (mmHg)
        random.randint(60, 110),                   # diastolic_bp (mmHg)
        round(random.uniform(95.0, 105.0), 1),     # temperature_f
        random.choice(["normal", "elevated", "critical"])  # status
    )
    events.append(row)

    # Exact copy: device re-sends the SAME event (same event_id, same event_time)
    if i in copy_positions:
        events.append(row)

    # Re-sent within watermark: device re-sends with SAME event_id but event_time + 1-5 min
    if i in within_wm_positions:
        resend_time = event_time + timedelta(minutes=random.randint(1, 5))
        events.append((row[0], row[1], resend_time, row[3], row[4], row[5], row[6], row[7]))

    # Re-sent OUTSIDE watermark: same event_id but event_time ~15h later (stored separately)
    if i in outside_wm_positions:
        resend_minute = 930 + (i // 5 - 1)  # 930, 932, 934, 936, 938 min after midnight
        resend_time = base_dt + timedelta(minutes=resend_minute)
        outside_resends.append((row[0], row[1], resend_time, row[3], row[4], row[5], row[6], row[7]))

vitals_schema = "event_id INT, patient_id INT, event_time TIMESTAMP, heart_rate INT, systolic_bp INT, diastolic_bp INT, temperature_f DOUBLE, status STRING"
spark.createDataFrame(events, schema=vitals_schema) \
    .write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.vital_sign_events")

# Store outside-wm re-sent events in a separate table (inserted during Demo 7)
spark.createDataFrame(outside_resends, schema=vitals_schema) \
    .write.mode("overwrite").saveAsTable(f"{FULL_SCHEMA}.vital_sign_resend")

# --- Summary ---
print(f"✅ Schema: {FULL_SCHEMA}")
print(f"✅ Volume: {VOLUME}")
print(f"   Patients:         {spark.table(f'{FULL_SCHEMA}.patients').count():>5}")
print(f"   Encounters:       {spark.table(f'{FULL_SCHEMA}.encounters').count():>5}")
print(f"   Claims:           {spark.table(f'{FULL_SCHEMA}.claims').count():>5}")
print(f"   Lab results:      {spark.table(f'{FULL_SCHEMA}.lab_results').count():>5}")
print(f"   Vital sign events:{spark.table(f'{FULL_SCHEMA}.vital_sign_events').count():>5}  (1,020 rows: 1,000 originals + 20 duplicates)")
print(f"   Resend events:     {spark.table(f'{FULL_SCHEMA}.vital_sign_resend').count():>5}  (5 outside-watermark re-sent, used in Demo 7)")
print(f"   FHIR resources:   {spark.read.json(f'{VOLUME}/fhir_resources').count():>5}")
print(f"\nStreaming source: {FULL_SCHEMA}.vital_sign_events (1,020 rows)")
print(f"  - 20 late events every 50th row (event_time 30 min behind)")
print(f"  - 10 exact copies (same event_id, same event_time)")
print(f"  - 10 re-sent within watermark (same event_id, event_time + 1-5 min)")
print(f"Resend table: {FULL_SCHEMA}.vital_sign_resend (5 rows)")
print(f"  - 5 re-sent outside watermark (same event_id, event_time ~15h later)")
print()
display(spark.table(f"{FULL_SCHEMA}.vital_sign_events").limit(5))

## 3.1 What is Structured Streaming?

**Structured Streaming** is a stream-processing engine built on top of the Spark SQL engine. You treat a data stream as an **unbounded table** — a table that keeps growing as new rows arrive. You write the same DataFrame or SQL operations you already know (`select`, `where`, `groupBy`, `join`), and the engine runs them continuously as new data arrives.

### Why it exists

| Problem without it | How Structured Streaming solves it |
|---|---|
| Writing separate batch and streaming code is error-prone | **One API** — same DataFrame/SQL code works for both batch and streaming |
| Crash mid-job means lost progress or duplicate writes | **Checkpointing** stores offsets + state + committed writes for exactly-once recovery |
| Late-arriving events corrupt window aggregations | **Watermarks** define how late is too late, bounding state and dropping stale events |
| Hard to control how often results are emitted | **Triggers** let you choose: real-time, every N seconds, or process-all-then-stop |

### When to use it vs alternatives

| Use case | Tool |
|---|---|
| Process new Delta rows continuously (near-real-time) | **Structured Streaming** |
| Process all data in scheduled batches (every hour/day) | **Batch DataFrame / SQL** |
| Complex pipeline with bronze/silver/gold layers, expectations, auto-retries | **Lakeflow Declarative Pipelines** (formerly DLT) |
| Very low-latency (< 1 ms) continuous processing | **Continuous processing** trigger (rare; has limitations) |

### How a streaming query works (micro-batch model)

```
  Source table         Spark engine (micro-batch)        Sink
  ┌──────────┐   ┌──────────────────────────┐    ┌──────────┐
  │ Delta /   │──▶│ Read new rows since last  │──▶│ Delta /  │
  │ Kafka /   │   │ offset → transform →     │    │ Memory / │
  │ Files     │   │ write batch → checkpoint │    │ Kafka    │
  └──────────┘   └──────────────────────────┘    └──────────┘
         ▲                                           │
         └──── offset tracked in checkpoint ──────────┘
```

> **Key idea:** Each micro-batch reads only the **new rows** since the last checkpoint, processes them, writes to the sink, and saves progress. If the query crashes and restarts, it picks up from the last checkpoint — no duplicates, no data loss.

### SQL equivalent (Lakeflow Declarative Pipelines only)

In a plain notebook, streaming APIs are PySpark only. Inside a Lakeflow Declarative Pipeline (formerly DLT), you can use SQL:
```sql
-- "STREAM()" converts a static table into a streaming source
SELECT * FROM STREAM(my_table);
```

## 3.2 readStream / writeStream and Trigger Types

### The basic pattern

```
  readStream          DataFrame ops          writeStream
  ┌────────┐          ┌──────────┐          ┌────────────┐
  │ Source  │──▶ .readStream ──▶│ filter,  │──▶ .writeStream ──▶│ Sink       │
  │ Delta,  │          │ groupBy, │          │ Delta, mem,│
  │ Kafka,  │          │ join …   │          │ console …  │
  └────────┘          └──────────┘          └────────────┘
```

### Trigger types — controlling how often a micro-batch runs

| Trigger | What it does | When to use | Self-terminates? |
|---|---|---|---|
| *(default)* | Runs as fast as possible (~0 ms interval) | Near-real-time, lowest latency | No |
| `processingTime="5 seconds"` | Runs a micro-batch every 5 seconds | Control batch frequency / cost | No |
| `availableNow=True` | Processes ALL available data in multiple batches, then stops | Batch-like jobs, backfill, scheduled ETL | **Yes** |
| `once=True` | Processes ONE batch, then stops | *(Legacy — prefer `availableNow`)* | Yes |
| `continuous="1 second"` | Continuous low-latency (experimental, limited ops) | Sub-second latency needs | No |

> **Exam tip:** `availableNow` is the modern replacement for `once`. `availableNow` can split work across **multiple** micro-batches (better for large backfills), while `once` processes only a **single** batch.

### PySpark syntax
```python
spark.readStream.format("delta").table("my_table")       # read from a Delta table as a stream
    .writeStream
    .trigger(availableNow=True)                           # process everything, then stop
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/.../ckpt")
    .start()                                               # launches the query
```

### SQL syntax (Lakeflow Declarative Pipelines)
```sql
-- Triggers are set in the pipeline settings, not in SQL itself.
-- In the pipeline definition, you configure the trigger as a pipeline property.
```

## 3.3 Output Modes — `append`, `update`, `complete`

Output mode controls **which rows the sink receives** at the end of each micro-batch.

| Mode | What the sink gets each trigger | Works with aggregation? | Needs watermark? | Typical use |
|---|---|---|---|---|
| **append** | Only **new** rows that will never change | Only aggregations **with watermark** (finalized windows) | **Yes** (for aggregations) | Writing new events to Delta |
| **update** | Only rows that **changed** since last trigger | Yes (all aggregations) | No | Dashboards, real-time updates |
| **complete** | The **entire** result set every trigger | Yes (aggregations **only** — not for plain selects) | No | Small aggregations, in-memory sinks |

### Rules to memorize

1. **No aggregation → `append` only.** A simple `select`/`where` has no state; rows just flow through.
2. **Aggregation + `append` → requires watermark.** Without a watermark, Spark cannot know when a window is “final” and would need unbounded state.
3. **`complete` mode outputs ALL rows every trigger** — fine for small result sets, dangerous for large ones (rewrites everything each batch).
4. **`update` mode outputs only changed rows** — efficient for large aggregations, but the sink must support updates (Delta does via MERGE).
5. **`complete` does NOT work with non-aggregation queries.**

### Decision flow

```
  Does the query have an aggregation (groupBy, count, sum, etc.)?
     │
     ├── NO  ──▶ append only
     │
     └── YES ──▶ Is the result set small enough to rewrite every batch?
                  │
                  ├── YES ──▶ complete  (e.g., count by status → 3 rows)
                  │
                  └── NO  ──▶ Do you need only-changed rows?
                               │
                               ├── YES ──▶ update
                               │
                               └── NO  ──▶ append (requires watermark + window)
```

## 3.4 Watermark — Bounding State and Handling Late Events

### What is a watermark?

A **watermark** is a moving time threshold. It tells Spark: “I do not expect any event with `event_time` older than this threshold to arrive.” It is computed as:

```
  watermark = MAX(event_time seen so far)  −  delay_threshold
```

### Why it exists (two problems it solves)

| Problem | How watermark solves it |
|---|---|
| State for window aggregations grows forever (all windows kept indefinitely) | Windows older than the watermark are **finalized and discarded** — state is bounded |
| `append` mode cannot emit aggregation results because it does not know when a window is “done” | The watermark marks windows as “final” so `append` mode can emit them |

### What happens to late events?

```
  Event timeline (event_time):
  ─────[window W1]────[window W2]────[window W3]────▶ now (MAX event_time)
                                           ▲
                                    watermark = now − delay
  ✓ Events within W3 (after watermark) → processed normally
  ✗ Events in W1 (before watermark)   → DROPPED (too late)
  ~ Events near the watermark           → MAY still be processed (not guaranteed dropped)
```

### PySpark syntax
```python
stream_df = (spark.readStream
    .format("delta").table("vitals")
    .withWatermark("event_time", "20 minutes")      # 20-minute delay tolerance
    .groupBy(window("event_time", "10 minutes"))     # 10-minute tumbling windows
    .count()
)
```

### SQL syntax (Lakeflow Declarative Pipelines)
```sql
-- Watermark is applied in PySpark for plain streaming.
-- In Lakeflow Declarative Pipelines (formerly DLT), the concept is the same
-- but handled via the Python API or pipeline configuration.
```

> **Exam tip:** A watermark does **not** guarantee every late event is dropped. Spark may still process some events that arrive later than the threshold. The guarantee is that state is **bounded** and windows are **eventually finalized**.

## 3.5 Checkpoint Location and foreachBatch

### Checkpoint location — what it stores and why

A **checkpoint location** is a durable path (UC volume) where Spark saves streaming progress. It stores:

| Component | Purpose |
|---|---|
| **Offsets** | Which rows from the source have already been read |
| **State** | Aggregation state, dedup state, etc. (for stateful queries) |
| **Commit log** | Which batches have been successfully written to the sink |

### How it enables exactly-once

```
  Crash
    │
    ▼
  Restart with same checkpointLocation
    │
    ├── Offset log says “last read = offset X” ──▶ resume from X+1 (no re-read)
    ├── State restored ──▶ aggregation state continues
    └── Commit log says “last committed = batch Y” ──▶ skip Y, continue (no re-write)
    
  Result: no duplicates, no data loss = exactly-once
```

> **Critical:** The checkpoint location MUST be durable and unique per query. Never use `/tmp` or DBFS root. Always use a **UC volume** path. Two queries sharing the same checkpoint will corrupt each other.

### foreachBatch — custom write logic per micro-batch

`foreachBatch(func)` gives you a **function** called for each micro-batch with two arguments:
- `batch_df` — the output rows for that batch (a static DataFrame)
- `batch_id` — a monotonically increasing batch number (useful for idempotent writes)

**When to use it** (instead of a built-in sink like Delta):

| Use case | Why foreachBatch |
|---|---|
| MERGE into a Delta table (upsert) | Built-in Delta sink only appends; MERGE needs custom logic |
| Write to multiple sinks | One batch → two destinations |
| Idempotent writes to external systems | Use `batch_id` to deduplicate on retry |

### PySpark pattern for MERGE via foreachBatch
```python
# Define the merge function
def upsert(batch_df, batch_id):
    batch_df.createOrReplaceTempView("batch_view")
    spark.sql("""
        MERGE INTO target t
        USING batch_view b
        ON t.patient_id = b.patient_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """
    )

# Use it in the streaming query
(spark.readStream.format("delta").table("source")
    .writeStream
    .foreachBatch(upsert)                      # call our function per batch
    .option("checkpointLocation", vol_path)
    .outputMode("update")
    .start())
```

> **Exam tip:** `foreachBatch` is NOT supported with the `continuous` trigger — only with micro-batch triggers (default, `processingTime`, `availableNow`).

## 3.6 Exam Traps ☠️

Review these common confusions before moving on:

| # | Trap | Truth |
|---|---|---|
| 1 | “`append` mode works with any aggregation” | **No** — `append` + aggregation requires a **watermark** (Spark needs to know when windows are final) |
| 2 | “`complete` mode is the most efficient” | **No** — it rewrites the **entire** result every trigger; fine for 3 rows, terrible for 3 million |
| 3 | “Watermark drops every late event” | **No** — watermark *bounds state*; some late events near the threshold may still be processed |
| 4 | “`once` and `availableNow` are the same” | **No** — `once` processes one batch; `availableNow` can process in multiple batches then stop |
| 5 | “`foreachBatch` works with continuous trigger” | **No** — `foreachBatch` only works with **micro-batch** triggers |
| 6 | “`dropDuplicates` without watermark is fine” | **No** — without a watermark, dedup state grows **unbounded** |
| 7 | “Two streams can share a checkpoint location” | **No** — sharing corrupts offsets; each query needs a **unique** checkpoint path |
| 8 | “`update` mode only works with aggregations” | **No** — `update` works with **any** query; it just outputs changed rows |
| 9 | “Checkpoint stores only offsets” | **No** — it stores offsets **+ state + commit log** |
| 10 | “Delta sink always supports updates” | **No** — the built-in Delta sink appends; for updates you need `foreachBatch` + MERGE |
| 11 | “`dropDuplicatesWithinWatermark` works without a watermark” | **No** — it **requires** `withWatermark()` first; otherwise it errors at query start |
| 12 | "Include `event_time` in the dedup key for safety" | **No** — if `event_time` is in the key, re-sent events with a different timestamp are treated as new rows. Dedup on the **event identifier only** |

## 4. Demos

Each demo has three parts:
1. A **Predict** cell — think about what the code will return *before* running it.
2. A **code** cell — every line is commented.
3. An **explanation** cell — what actually happened and why.

> **Serverless reminder:** On serverless compute, all streaming queries must use `trigger(availableNow=True)` or `trigger(once=True)` — the default `processingTime` trigger is not supported. A `checkpointLocation` is also required (no implicit temp checkpoints).

### Demo 1 — Predict: readStream basics

Before running the cell below, predict:
1. How many rows will be in the `demo1_basics` in-memory table?
2. Will the streaming query still be running after `awaitTermination` completes?
3. What output mode is required here — and why not `complete` or `update`?

> Think about it, then run the cell to check.

In [0]:
%python
# Demo 1: The simplest streaming query — read from Delta, write to memory
# Key pattern: readStream → writeStream → availableNow → checkpoint

query1 = (spark
    .readStream                                  # Start a streaming read
    .format("delta")                             # Source format: Delta Lake
    .table(f"{FULL_SCHEMA}.vital_sign_events")   # Read this table as a stream
    .writeStream                                 # Start a streaming write
    .format("memory")                            # Sink: in-memory table (fast for demos)
    .queryName("demo1_basics")                   # Name of the in-memory table
    .outputMode("append")                        # No aggregation → append is the only valid mode
    .trigger(availableNow=True)                  # Process all available data, then stop
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo1")  # Required on serverless
    .start())                                    # Launch the query

query1.awaitTermination(timeout=60)              # Wait for availableNow to finish
print(f"Query active after completion: {query1.isActive}")  # Should be False
print(f"Rows in sink: {spark.table('demo1_basics').count()}")
display(spark.table("demo1_basics").limit(5))

**Result explained:**
1. The sink has all rows from `vital_sign_events` (1,000 rows after a fresh setup run).
2. `awaitTermination` returns when the query finishes. With `availableNow=True`, the query processes all data and **self-terminates**, so `isActive` is `False`.
3. `append` is the natural choice because there is **no aggregation** — rows simply pass through. `complete` only works with aggregations; `update` would also work but is unnecessary here.

> **Serverless note:** On serverless compute, you MUST specify `trigger(availableNow=True)` (or `once=True`) — the default `processingTime` trigger is not supported. You also MUST provide a `checkpointLocation`.

### Demo 2 — Predict: Trigger types

Before running, predict:
1. After `awaitTermination` completes, will `query.isActive` be `True` or `False`?
2. What happens if you try to use `trigger(processingTime="5 seconds")` on serverless compute?

> Think about it, then run the cell to check.

In [0]:
%python
# Demo 2: Trigger types — AvailableNow vs ProcessingTime
# availableNow: processes ALL data in one or more batches, then terminates
# processingTime: runs every N seconds (NOT supported on serverless — needs classic compute)

# --- AvailableNow trigger ---
query2 = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .writeStream
    .format("memory")
    .queryName("demo2_available_now")
    .outputMode("append")
    .trigger(availableNow=True)                  # Process everything, then stop
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo2")
    .start())

query2.awaitTermination(timeout=60)               # Wait for it to self-terminate
print(f"AvailableNow — Active after finish: {query2.isActive}")
print(f"AvailableNow — Rows processed: {spark.table('demo2_available_now').count()}")

# --- Try processingTime (will fail on serverless) ---
print("\nAttempting processingTime trigger on serverless...")
try:
    query2b = (spark
        .readStream
        .format("delta")
        .table(f"{FULL_SCHEMA}.vital_sign_events")
        .writeStream
        .format("memory")
        .queryName("demo2b_processing_time")
        .outputMode("append")
        .trigger(processingTime="5 seconds")     # NOT supported on serverless
        .option("checkpointLocation", f"{VOLUME}/checkpoints/demo2b")
        .start())
except Exception as e:
    error_msg = str(e)
    for line in error_msg.split('\n'):
        if 'NOT_SUPPORTED' in line or 'not supported' in line.lower():
            print(f"Expected error: {line.strip()}")
            break
    print("→ processingTime requires classic compute, not serverless")

**Result explained:**
1. `availableNow` processes all data and **self-terminates** → `isActive` is `False`.
2. `processingTime` is **not supported on serverless** compute — you need classic compute for continuous micro-batch triggers. Use `availableNow` for batch-like jobs on serverless; use `processingTime` for near-real-time on classic compute.

| Trigger | Serverless | Classic compute | Self-terminates |
|---|---|---|---|
| `availableNow=True` | ✅ | ✅ | Yes |
| `processingTime="5s"` | ❌ | ✅ | No |
| Default (none) | ❌ | ✅ | No |
| `once=True` | ✅ | ✅ | Yes (legacy — prefer `availableNow`) |

### Demo 3 — Predict: Output modes with aggregation

Before running, predict:
1. With `complete` mode, how many rows will be in the sink after processing?
2. If you use `append` mode with a `groupBy` but **no watermark**, what will happen?
3. If you use `complete` mode with a plain `select` (no aggregation), what will happen?

> Think about it, then run the cell to check.

In [0]:
%python
# Demo 3: Output modes with aggregations
# Test 3a: complete mode with groupBy (should work — small result set)
query3a = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .groupBy("status")                           # Group by status (3 groups: normal, elevated, critical)
    .count()                                     # Count per status
    .writeStream
    .format("memory")
    .queryName("demo3a_complete")
    .outputMode("complete")                      # Full result every trigger (fine for 3 rows)
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo3a")
    .start())
query3a.awaitTermination(timeout=60)
print("=== COMPLETE mode with groupBy ===")
display(spark.table("demo3a_complete"))

# Test 3b: append mode with groupBy but NO watermark (should ERROR)
print("\n=== APPEND mode with groupBy, NO watermark ===")
try:
    query3b = (spark
        .readStream
        .format("delta")
        .table(f"{FULL_SCHEMA}.vital_sign_events")
        .groupBy("status")
        .count()
        .writeStream
        .format("memory")
        .queryName("demo3b_append_no_wm")
        .outputMode("append")                    # Append + aggregation + no watermark = ERROR
        .trigger(availableNow=True)
        .option("checkpointLocation", f"{VOLUME}/checkpoints/demo3b")
        .start())
except Exception as e:
    error_text = str(e)
    if "watermark" in error_text.lower() or "Append" in error_text:
        print("Expected error: append mode with aggregation requires a watermark!")
        print("  → Fix: add .withWatermark('event_time', '20 minutes') before groupBy")
    else:
        print(f"Error: {error_text[:200]}")

**Result explained:**
1. `complete` mode outputs all 3 status groups (`normal`, `elevated`, `critical`) with their counts. Every trigger rewrites the full result — fine for 3 rows, wasteful for millions.
2. `append` mode with `groupBy` and **no watermark** → **error**. Spark cannot know when a group is "done" without a watermark, so it refuses to start. Fix: add `.withWatermark("event_time", "20 minutes")` and use a `window()` in `groupBy`.
3. `complete` mode with a plain `select` (no aggregation) → also **error** — `complete` only works with aggregations.

| Output mode | No aggregation | Aggregation (no watermark) | Aggregation + watermark |
|---|---|---|---|
| `append` | ✅ | ❌ error | ✅ |
| `update` | ✅ | ✅ | ✅ |
| `complete` | ❌ error | ✅ | ✅ |

### Demo 4 — Predict: Watermark and window aggregation

Before running, predict:
1. How many 10-minute windows will be emitted in `append` mode with a 20-minute watermark?
2. The source has ~1,000 events spanning ~1,000 minutes. How many 10-minute windows would exist without watermark?
3. Which events are being dropped, and why?

> Think about it, then run the cell to check.

In [0]:
%python
from pyspark.sql.functions import window  # Import the window function for time-based grouping

# Demo 4: Watermark + window aggregation with append mode
# The watermark allows append mode to finalize and emit completed windows.
# Late events (event_time 30 min behind) that fall before the watermark are dropped.

query4 = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .withWatermark("event_time", "20 minutes")   # 20-minute delay tolerance
    .groupBy(window("event_time", "10 minutes"))  # 10-minute tumbling windows
    .count()                                     # Count events per window
    .writeStream
    .format("memory")
    .queryName("demo4_watermark")
    .outputMode("append")                        # Append requires watermark for aggregations
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo4")
    .start())
query4.awaitTermination(timeout=60)

window_count = spark.table("demo4_watermark").count()
print(f"Windows emitted (with watermark): {window_count}")
print(f"Expected without watermark: ~100 windows (1000 min / 10 min)")
print(f"Difference: late events in some windows were dropped by the watermark\n")
display(spark.table("demo4_watermark").orderBy("window.start").limit(10))

**Result explained:**
1. About **97 windows** are emitted (vs ~100 without watermark). The missing windows contain late events that fell before the watermark and were dropped.
2. Without watermark, all ~100 windows would need unbounded state — Spark would keep every window open indefinitely.
3. Events with `event_time` more than 20 minutes behind the latest seen `event_time` are dropped. The late events in our data (every 50th row, 30 min behind) exceed this threshold, so some are dropped.

**The watermark does two things at once:**
- **Bounds state:** old windows are finalized and discarded (memory freed)
- **Enables `append` mode:** finalized windows can be emitted because they won't change again

### Demo 5 — Predict: foreachBatch with MERGE

Before running, predict:
1. Why can't we use the built-in Delta sink (`.format("delta")`) for an upsert?
2. What does `batch_id` represent in the `foreachBatch` function?
3. After the query finishes, how many rows will be in the target table?

> Think about it, then run the cell to check.

In [0]:
%python
# Demo 5: foreachBatch with MERGE into a Delta table
# The built-in Delta sink only APPENDS. For upserts (MERGE), we need foreachBatch.

# Create a fresh target table for patient vitals summary
spark.sql(f"DROP TABLE IF EXISTS {FULL_SCHEMA}.demo5_patient_summary")
spark.sql(f"""
    CREATE TABLE {FULL_SCHEMA}.demo5_patient_summary (
        patient_id INT,
        avg_heart_rate DOUBLE,
        reading_count BIGINT
    )
""")

# Define the MERGE function — called once per micro-batch
def upsert_patient_vitals(batch_df, batch_id):
    """Merge each batch into the target: update existing patients, insert new ones."""
    batch_df.createOrReplaceTempView("batch_vitals")  # Make batch queryable via SQL
    spark.sql(f"""
        MERGE INTO {FULL_SCHEMA}.demo5_patient_summary t
        USING (
            SELECT patient_id,
                   AVG(heart_rate) AS avg_heart_rate,
                   COUNT(*) AS reading_count
            FROM batch_vitals
            GROUP BY patient_id
        ) s
        ON t.patient_id = s.patient_id
        WHEN MATCHED THEN UPDATE SET
            t.avg_heart_rate = s.avg_heart_rate,
            t.reading_count = t.reading_count + s.reading_count
        WHEN NOT MATCHED THEN INSERT (patient_id, avg_heart_rate, reading_count)
        VALUES (s.patient_id, s.avg_heart_rate, s.reading_count)
    """)

# Start the streaming query with foreachBatch
query5 = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .writeStream
    .foreachBatch(upsert_patient_vitals)          # Custom write logic per batch
    .outputMode("update")                        # Update: only changed rows per batch
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo5")
    .start())
query5.awaitTermination(timeout=60)

target_count = spark.table(f"{FULL_SCHEMA}.demo5_patient_summary").count()
print(f"Patient summaries in target: {target_count}")
print(f"batch_id in foreachBatch: monotonically increasing per micro-batch (0, 1, 2, ...)")
display(spark.table(f"{FULL_SCHEMA}.demo5_patient_summary").orderBy("patient_id").limit(10))

**Result explained:**
1. The built-in Delta sink only **appends** — it cannot update existing rows. `foreachBatch` gives you a static DataFrame per batch, so you can run any SQL (including `MERGE`) against it.
2. `batch_id` is a monotonically increasing number (0, 1, 2, ...) assigned to each micro-batch. It's useful for idempotent writes: if a batch fails and retries, the same `batch_id` lets you detect and skip duplicates.
3. The target has **50 rows** (one per patient). Each patient's `reading_count` reflects how many vital-sign events they had, and `avg_heart_rate` shows their average.

> **Key takeaway:** Use `foreachBatch` + `MERGE` when you need upserts. Use the built-in Delta sink when append-only is sufficient.

### Demo 6 — Predict: Checkpoint recovery

Before running, predict:
1. After the first run processes all data, how many rows will be in the Delta sink?
2. After inserting 50 new rows and restarting with the same checkpoint, how many **new** rows will be processed?
3. What would happen if you started the second run with a **different** checkpoint location?

> Think about it, then run the cell to check.

In [0]:
%python
import shutil  # Python file system utilities for cleanup
import random
from datetime import datetime, timedelta

# Demo 6: Checkpoint recovery — exactly-once semantics
# Step 1: Run a stream that processes all existing data into a Delta sink
# Step 2: Insert 50 new rows into the source
# Step 3: Restart with the SAME checkpoint — only new rows should be processed

CKPT6 = f"{VOLUME}/checkpoints/demo6_recovery"      # Checkpoint path (must be unique per query)
SINK6 = f"{VOLUME}/sinks/demo6_recovery"             # Delta sink path

# Clean up any previous test artifacts (idempotent demo)
for p in [CKPT6, SINK6]:
    try: shutil.rmtree(p)                           # Remove old checkpoint and sink
    except: pass                                    # Ignore if doesn't exist

# --- Step 1: First run — process existing data ---
query6a = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .writeStream
    .format("delta")                                # Delta sink (supports checkpoint recovery)
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CKPT6)
    .start(SINK6))                                  # Write to this Delta path
query6a.awaitTermination(timeout=60)
run1_count = spark.read.format("delta").load(SINK6).count()
print(f"Run 1: sink has {run1_count} rows")

# --- Step 2: Insert 50 new rows into the source ---
random.seed(77)
base_dt = datetime(2024, 9, 22, 0, 0, 0)
max_event_id = spark.table(f"{FULL_SCHEMA}.vital_sign_events").agg({"event_id": "max"}).collect()[0][0]
new_rows = []
for i in range(max_event_id + 1, max_event_id + 51):
    event_time = base_dt + timedelta(minutes=i)
    new_rows.append((i, random.choice(range(1001, 1051)), event_time,
        random.randint(50, 120), random.randint(80, 180), random.randint(60, 110),
        round(random.uniform(95.0, 105.0), 1), random.choice(["normal", "elevated", "critical"])))
new_df = spark.createDataFrame(new_rows,
    "event_id INT, patient_id INT, event_time TIMESTAMP, heart_rate INT, systolic_bp INT, diastolic_bp INT, temperature_f DOUBLE, status STRING")
new_df.write.mode("append").saveAsTable(f"{FULL_SCHEMA}.vital_sign_events")
print(f"Inserted 50 new rows. Source now has {spark.table(f'{FULL_SCHEMA}.vital_sign_events').count()} rows")

# --- Step 3: Restart with SAME checkpoint — only new rows processed ---
query6b = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CKPT6)             # SAME checkpoint → resume from last offset
    .start(SINK6))
query6b.awaitTermination(timeout=60)
run2_count = spark.read.format("delta").load(SINK6).count()
new_processed = run2_count - run1_count
print(f"Run 2: sink has {run2_count} rows (new rows processed = {new_processed})")
print(f"\n✅ Checkpoint recovery verified: exactly {new_processed} new rows, no reprocessing!")

**Result explained:**
1. Run 1 processes all existing rows into the Delta sink (e.g., 1,000 rows).
2. After inserting 50 new rows and restarting with the **same checkpoint**, only **50 new rows** are processed. The checkpoint stored the offset from Run 1, so the query resumes exactly where it left off.
3. If you started with a **different** checkpoint, the query would treat itself as a **new** query and **reprocess all data** from the beginning — causing duplicates in the sink.

> **Critical:** Each streaming query must have its **own unique checkpoint location**. Sharing checkpoints between queries corrupts the offset tracking. The checkpoint stores three things: **offsets** (what was read), **state** (aggregation progress), and **commit log** (what was written).

### Demo 7 — Predict: Streaming deduplication

Before running, predict:
The source has 1,020 rows: 1,000 unique `event_id`s plus 20 duplicates (10 exact copies + 10 re-sent with a different `event_time`). 5 more re-sent events are stored separately.

1. Approach (a): `dropDuplicates(["event_id"])` without watermark — how many rows survive?
2. Approach (b): `withWatermark("event_time", "80 minutes") + dropDuplicates(["event_id", "event_time"])` — will re-sent events (same `event_id`, different `event_time`) be caught?
3. Approach (c): `withWatermark("event_time", "80 minutes") + dropDuplicatesWithinWatermark(["event_id"])` — two runs. Run 1 catches all 20 duplicates. Run 2 inserts 5 re-sent events whose originals have been evicted by the watermark. Will they be caught?
4. What happens if you call `dropDuplicatesWithinWatermark()` **without** first calling `withWatermark()`?

> Think about it, then run the cell to check.

In [0]:
%python
# Demo 7: Streaming deduplication — three approaches compared
# Source: 1,020 rows = 1,000 unique event_ids + 10 exact copies + 10 re-sent (different event_time)
# Separate table: 5 outside-watermark re-sent events (used in approach c, Run 2)
import shutil

# --- Approach (a): dropDuplicates WITHOUT watermark (unbounded state) ---
# Dedup on event_id only. Catches ALL duplicates (exact copies AND re-sent).
# ⚠️ State grows forever — fine for a demo, dangerous in production.
query7a = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .dropDuplicates(["event_id"])
    .writeStream
    .format("memory")
    .queryName("demo7a_no_watermark")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo7a")
    .start())
query7a.awaitTermination(timeout=60)
cnt_a = spark.table("demo7a_no_watermark").count()

# --- Approach (b): withWatermark + dropDuplicates(["event_id", "event_time"]) ---
# ✅ Bounded state, BUT event_time is in the dedup key!
# Re-sent events have a DIFFERENT event_time → treated as NEW events → NOT caught.
query7b = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .withWatermark("event_time", "80 minutes")
    .dropDuplicates(["event_id", "event_time"])
    .writeStream
    .format("memory")
    .queryName("demo7b_event_time_in_key")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/demo7b")
    .start())
query7b.awaitTermination(timeout=60)
cnt_b = spark.table("demo7b_event_time_in_key").count()

# --- Approach (c): withWatermark + dropDuplicatesWithinWatermark(["event_id"]) ---
# ✅ Bounded state, dedup on event_id ONLY (no event_time in the key).
# Catches re-sent duplicates within the watermark delay.
# To show that events outside the watermark are MISSED, we need two runs:
#   Run 1: process all 1,020 rows → all 20 duplicates caught (state still active)
#   Insert 5 re-sent events (same event_id, event_time ~15h later)
#   Run 2: originals have been evicted by the watermark → 5 re-sent NOT caught → 1,005 total
ckpt7c = f"{VOLUME}/checkpoints/demo7c"
sink7c = f"{VOLUME}/sinks/demo7c"
shutil.rmtree(ckpt7c, ignore_errors=True)
dbutils.fs.rm(sink7c, True)

query7c1 = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .withWatermark("event_time", "80 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
    .writeStream
    .format("delta")
    .option("path", sink7c)
    .trigger(availableNow=True)
    .option("checkpointLocation", ckpt7c)
    .start())
query7c1.awaitTermination(timeout=60)
cnt_c1 = spark.read.format("delta").load(sink7c).count()

# Insert the 5 outside-watermark re-sent events into the source
spark.sql(f"INSERT INTO {FULL_SCHEMA}.vital_sign_events SELECT * FROM {FULL_SCHEMA}.vital_sign_resend")

# Run 2: same checkpoint — only the 5 new rows are processed.
# The originals' event_ids have been evicted (their event_time is behind the watermark),
# so the 5 re-sent events are NOT detected as duplicates.
query7c2 = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    .withWatermark("event_time", "80 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
    .writeStream
    .format("delta")
    .option("path", sink7c)
    .trigger(availableNow=True)
    .option("checkpointLocation", ckpt7c)
    .start())
query7c2.awaitTermination(timeout=60)
cnt_c = spark.read.format("delta").load(sink7c).count()

print(f"Source: 1,020 rows (1,000 unique event_ids + 20 duplicates)")
print(f"         + 5 outside-watermark re-sent (inserted for Run 2)")
print()
print(f"(a) dropDuplicates(['event_id']) no watermark:       {cnt_a} rows  ← all duplicates caught")
print(f"(b) dropDuplicates(['event_id','event_time']) + wm:   {cnt_b} rows  ← re-sent MISSED (different event_time)")
print(f"(c) dropDuplicatesWithinWatermark(['event_id']) + wm: {cnt_c} rows  ← within-wm caught, outside-wm MISSED")
print()
print("Key insight: (b) includes event_time in the dedup key, so re-sent events")
print("with a DIFFERENT event_time are treated as new events. (c) deduplicates on")
print("event_id ONLY within the watermark window, catching re-sent duplicates whose")
print("originals are still in state — but missing those whose originals were evicted.")

**Result explained:**

| Approach | Rows | Why |
|---|---|---|
| (a) `dropDuplicates(["event_id"])` no watermark | **1,000** | Dedup on `event_id` only. Catches all 20 duplicates (exact copies + re-sent). **Unbounded state** — keeps every seen `event_id` forever. |
| (b) `withWatermark("event_time", "80 min") + dropDuplicates(["event_id", "event_time"])` | **1,010** | `event_time` is in the dedup key. Re-sent events have a **different** `event_time` → treated as new rows → **10 re-sent duplicates missed**. Only 10 exact copies caught. State is bounded. |
| (c) `withWatermark("event_time", "80 min") + dropDuplicatesWithinWatermark(["event_id"])` | **1,005** | Dedup on `event_id` only (no `event_time` in key). Run 1 catches all 20 duplicates. Run 2 inserts 5 re-sent events whose originals were **evicted by the watermark** → not detected → 5 extra rows. State is bounded. |

> **Why (b) misses re-sent duplicates:** Including `event_time` in the dedup key means that a re-sent event (same `event_id`, different `event_time`) has a different key tuple and is treated as a brand-new row. This is a common mistake — the dedup key should contain only the **event identifier**, not the timestamp.
>
> **Why (c) misses outside-watermark re-sent events:** `dropDuplicatesWithinWatermark` deduplicates on `event_id` within the watermark window. When a duplicate arrives after the watermark has evicted the original's `event_id` from state, the duplicate is **not detected** — the original is gone. This is the trade-off of bounded state: very-late duplicates may slip through.
>
> **Key distinction — `dropDuplicates` vs `dropDuplicatesWithinWatermark`:**
> - `dropDuplicates` + watermark: the watermark bounds state but the **semantics are the same** as without watermark — just with eviction.
> - `dropDuplicatesWithinWatermark`: exists specifically for duplicates whose **event time differs** from the original (e.g., a device re-sends the same event with a slightly different timestamp). It deduplicates on the key columns only, using the watermark column to decide when to evict state — the **watermark column does not need to be in the key**.
>
> **Exam trap:** `dropDuplicates` on a streaming DataFrame WITHOUT a watermark produces **unbounded state**. `dropDuplicatesWithinWatermark` **requires** `withWatermark()` first or it errors. Never put the event-time column in the dedup key — it defeats the purpose of deduplication.

## 5. Syntax Cheat Sheet

### PySpark patterns (plain notebooks) — memorize these

```python
# 1. Read a Delta table as a stream
stream_df = spark.readStream.format("delta").table("catalog.schema.table")

# 2. Write to a sink (memory, Delta, console, foreachBatch)
stream_df.writeStream \
    .format("memory") \
    .queryName("my_results") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "/Volumes/cat/sch/vol/ckpt") \
    .start()

# 3. Watermark + window aggregation (enables append mode)
stream_df \
    .withWatermark("event_time", "20 minutes") \
    .groupBy(window("event_time", "10 minutes")) \
    .count()

# 4. foreachBatch with MERGE
def upsert(batch_df, batch_id):
    batch_df.createOrReplaceTempView("src")
    spark.sql("MERGE INTO target t USING src s ON t.id = s.id WHEN MATCHED THEN UPDATE * WHEN NOT MATCHED THEN INSERT *")

stream_df.writeStream.foreachBatch(upsert).outputMode("update").start()

# 5. Deduplicate a stream (bounded state — always use withWatermark!)
#    Dedup on the EVENT IDENTIFIER only — never include event_time in the key!
stream_df.withWatermark("event_time", "20 minutes") \
    .dropDuplicatesWithinWatermark(["event_id"])  # catches re-sent dups within watermark
# ⚠️ .dropDuplicates(["event_id", "event_time"]) MISSES re-sent events (different event_time = different key)
# ⚠️ .dropDuplicates(["event_id"]) without watermark = UNBOUNDED state (memory leak)
```

### SQL patterns (Lakeflow Declarative Pipelines only)

```sql
-- Read a table as a stream inside a pipeline
SELECT * FROM STREAM(catalog.schema.table);

-- Create a streaming table (pipeline syntax)
CREATE STREAMING TABLE my_sink AS
  SELECT * FROM STREAM(catalog.schema.source);
```

### Quick reference table

| Concept | PySpark | SQL (pipelines only) |
|---|---|---|
| Read stream | `spark.readStream.format("delta").table(...)` | `FROM STREAM(table)` |
| Write stream | `.writeStream.format("memory").queryName(...).start()` | `CREATE STREAMING TABLE ...` |
| Trigger | `.trigger(availableNow=True)` | Pipeline setting |
| Output mode | `.outputMode("append")` | N/A (streaming table = append-like) |
| Watermark | `.withWatermark("event_time", "20 min")` | PySpark API in pipeline |
| foreachBatch | `.foreachBatch(func)` | N/A |
| Dedup (event_id only) | `.withWatermark(...).dropDuplicatesWithinWatermark(["event_id"])` | N/A |

## 6. Check Your Understanding

Answer these questions *before* scrolling to the solutions at the bottom.

1. You have a streaming query with `groupBy("status").count()` and `outputMode("append")`. The query fails to start. What is missing, and why is it required?

2. A checkpoint location stores three things. Name all three and explain what each is for.

3. You need to write streaming data into a Delta table, but existing rows must be **updated** (not appended). Which two things must you use together to achieve this?

## 7. Labs

> **Before starting the labs:** Re-run the **Classroom Setup** cell (Cell 2) to reset the source table to 1,000 rows.
>
> **Do not run the validation cells until you have completed the exercise.** The validation cell checks your work and prints `✅ Passed` or a hint about what is wrong.

### Lab Level 1 — Fill in the Blanks

**Task:** Build a streaming query that reads `vital_sign_events`, filters for high heart rate (> 100 bpm), and writes the results to a memory sink called `lab1_high_hr`.

There are 3 blanks (`<FILL_IN>`) to complete.

In [0]:
%python
# Lab Level 1: Filter streaming vitals for high heart rate alerts
# Task: Read the vital_sign_events stream, filter for heart_rate > 100,
#       and write to a memory sink called "lab1_high_hr".

from pyspark.sql.functions import col  # Import col for column references

lab1_query = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")      # Source table
    .filter(col("heart_rate") > <FILL_IN>)           # Filter for high heart rate (> 100 bpm)
    .select("event_id", "patient_id", "event_time", "heart_rate", "status")
    .writeStream
    .format("memory")
    .queryName(<FILL_IN>)                            # Name the sink "lab1_high_hr"
    .outputMode(<FILL_IN>)                           # No aggregation → which output mode?
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/lab1")
    .start())

lab1_query.awaitTermination(timeout=60)
lab1_count = spark.table("lab1_high_hr").count()
print(f"High heart rate alerts: {lab1_count}")
display(spark.table("lab1_high_hr").limit(10))

In [0]:
%python
# Validation for Lab Level 1 — run AFTER you complete the exercise above
try:
    lab1_result = spark.table("lab1_high_hr")
    lab1_count = lab1_result.count()

    if lab1_count == 0:
        print("❌ No rows returned. Check your filter — heart_rate should be > 100.")
    elif lab1_result.filter("heart_rate <= 100").count() > 0:
        print("❌ Some rows have heart_rate <= 100. Your filter threshold is wrong.")
    elif set(lab1_result.columns) != {"event_id", "patient_id", "event_time", "heart_rate", "status"}:
        print(f"❌ Column mismatch. Expected 5 columns, got: {lab1_result.columns}")
    else:
        print(f"✅ Passed! {lab1_count} high heart rate alerts found.")
except Exception as e:
    print(f"❌ Table 'lab1_high_hr' not found. Check your queryName. Error: {str(e)[:150]}")

### Lab Level 2 — Fix the Bug

**Task:** This streaming query counts vital events per 10-minute window and status. It has **two bugs** that prevent it from running on serverless compute.

Find and fix both bugs so the query runs successfully.

> **Hint:** One bug is about a missing piece for `append` mode with aggregations. The other is about a trigger type.

In [0]:
%python
# Lab Level 2: Fix the Bug
# This streaming query counts vital events per 10-minute window and status.
# There are TWO bugs that prevent it from running on serverless compute:
#   BUG 1: Something required by append mode for aggregations is missing
#   BUG 2: The trigger type is not supported on serverless compute
#
# Fix both issues so the query runs successfully.

from pyspark.sql.functions import window, col

lab2_query = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    # BUG 1: Something is missing here that append mode needs for aggregations...
    .groupBy(window("event_time", "10 minutes"), col("status"))
    .count()
    .writeStream
    .format("memory")
    .queryName("lab2_window_counts")
    .outputMode("append")
    .trigger(processingTime="5 seconds")            # BUG 2: This trigger doesn't work on serverless
    .option("checkpointLocation", f"{VOLUME}/checkpoints/lab2")
    .start())

lab2_query.awaitTermination(timeout=60)
lab2_count = spark.table("lab2_window_counts").count()
print(f"Window counts: {lab2_count}")
display(spark.table("lab2_window_counts").limit(10))

In [0]:
%python
# Validation for Lab Level 2 — run AFTER you fix the bugs above
try:
    lab2_result = spark.table("lab2_window_counts")
    lab2_count = lab2_result.count()

    if lab2_count == 0:
        print("❌ No rows returned. Two bugs to fix:")
        print("   1. Add .withWatermark('event_time', '20 minutes') before groupBy")
        print("   2. Change trigger to availableNow=True (not processingTime)")
    elif "window" not in lab2_result.columns or "status" not in lab2_result.columns:
        print("❌ Missing expected columns ('window', 'status'). Check your groupBy.")
    elif "count" not in lab2_result.columns:
        print("❌ Missing 'count' column. Check your aggregation.")
    else:
        print(f"✅ Passed! {lab2_count} window-status groups found.")
except Exception as e:
    error_text = str(e)
    if "watermark" in error_text.lower():
        print("❌ Append mode with aggregation needs a watermark.")
        print("   Add .withWatermark('event_time', '20 minutes') before groupBy.")
    elif "NOT_SUPPORTED" in error_text or "not supported" in error_text.lower():
        print("❌ The trigger is not supported on serverless.")
        print("   Use trigger(availableNow=True) instead of processingTime.")
    else:
        print(f"❌ Query failed. Two bugs to fix:")
        print("   1. Add .withWatermark('event_time', '20 minutes') before groupBy")
        print("   2. Change trigger to availableNow=True (not processingTime)")

### Lab Level 2b — Fix the Bug: Streaming deduplication

**Task:** This streaming query attempts to deduplicate vital-sign events by `event_time` using `dropDuplicatesWithinWatermark`. It has **one bug** that prevents it from running.

Find and fix the bug so the query runs successfully.

> **Hint:** `dropDuplicatesWithinWatermark` has a prerequisite that must be set *before* it is called. Look at Demo 7 for the pattern.

In [0]:
%python
# Lab Level 2b: Fix the Bug — Streaming deduplication
# This query tries to deduplicate vital events by event_time using
# dropDuplicatesWithinWatermark. There is ONE bug that prevents it from running.
#
# Fix the bug so the query runs successfully.

lab2b_query = (spark
    .readStream
    .format("delta")
    .table(f"{FULL_SCHEMA}.vital_sign_events")
    # BUG: dropDuplicatesWithinWatermark requires something to be set first...
    .dropDuplicatesWithinWatermark(["event_time"])
    .writeStream
    .format("memory")
    .queryName("lab2b_dedup")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", f"{VOLUME}/checkpoints/lab2b")
    .start())

lab2b_query.awaitTermination(timeout=60)
lab2b_count = spark.table("lab2b_dedup").count()
print(f"Deduplicated rows: {lab2b_count}")
display(spark.table("lab2b_dedup").limit(10))

In [0]:
%python
# Validation for Lab Level 2b — run AFTER you fix the bug above
try:
    lab2b_result = spark.table("lab2b_dedup")
    lab2b_count = lab2b_result.count()

    if lab2b_count == 0:
        print("❌ No rows returned. The query may have failed — check for errors above.")
    elif lab2b_count == 1000:
        print("❌ All 1000 rows returned — no duplicates removed. Did you dedup on 'event_time'?")
    elif lab2b_count > 990:
        print(f"⚠️  {lab2b_count} rows — some duplicates may remain. Expected ~980 unique event_times.")
    else:
        print(f"✅ Passed! {lab2b_count} unique rows (duplicates removed).")
except Exception as e:
    error_text = str(e)
    if "without watermark" in error_text.lower():
        print("❌ dropDuplicatesWithinWatermark requires a watermark.")
        print("   Add .withWatermark('event_time', '20 minutes') before dropDuplicatesWithinWatermark.")
    else:
        print("❌ Query failed. Check the error above.")
        print(f"   Error: {error_text[:200]}")

### Lab Level 3 — From Scratch

**Task:** Build a streaming query **from scratch** that:

1. Reads `vital_sign_events` as a stream (format `"delta"`, table `f"{FULL_SCHEMA}.vital_sign_events"`)
2. Filters for `status == "critical"` events only
3. Sets a **15-minute watermark** on `event_time`
4. Groups by **5-minute windows** and counts events
5. Writes to a memory sink named `"lab3_critical_windows"` in **append mode**
6. Uses `trigger(availableNow=True)` and `checkpointLocation` at `f"{VOLUME}/checkpoints/lab3"`

**Hints:**
- Import `window` from `pyspark.sql.functions`
- Use `.withWatermark("event_time", "15 minutes")` **before** `.groupBy()`
- Use `window("event_time", "5 minutes")` in `groupBy`
- Use `.filter(col("status") == "critical")` or `.where("status = 'critical'")`
- The `checkpointLocation` is required on serverless compute

In [0]:
%python
# Lab Level 3: From Scratch
#
# Write a streaming query that:
#   1. Reads vital_sign_events as a stream
#   2. Filters for status == "critical"
#   3. Sets a 15-minute watermark on event_time
#   4. Groups by 5-minute windows and counts
#   5. Writes to memory sink "lab3_critical_windows" in append mode
#   6. Uses trigger(availableNow=True) and checkpointLocation

# TODO: Import what you need

# TODO: Write your streaming query here

# TODO: Wait for it to finish (awaitTermination)

# TODO: Display your results

In [0]:
%python
# Validation for Lab Level 3 — run AFTER you write and execute your query above
try:
    lab3_result = spark.table("lab3_critical_windows")
    lab3_count = lab3_result.count()

    if lab3_count == 0:
        print("❌ No rows returned. Check:")
        print("   - Did you filter for status == 'critical'?")
        print("   - Did you use append mode (requires watermark)?")
        print("   - Did you use trigger(availableNow=True)?")
    elif "window" not in lab3_result.columns:
        print("❌ Missing 'window' column. Use window('event_time', '5 minutes') in groupBy.")
    elif "count" not in lab3_result.columns:
        print("❌ Missing 'count' column. Use .count() after groupBy.")
    else:
        print(f"✅ Passed! {lab3_count} critical windows found.")
        display(lab3_result.orderBy("window.start").limit(10))
except Exception as e:
    print(f"❌ Table 'lab3_critical_windows' not found or query failed.")
    print("   Check your queryName, trigger, and checkpointLocation.")
    print(f"   Error: {str(e)[:200]}")

## 8. Exam-Style Questions

Choose the **one best answer** for each question. Answers and explanations are at the bottom of the notebook.

---

**Q1.** A streaming query reads from a Delta source, groups by `window(event_time, "10 minutes")`, and uses `outputMode("append")`. The query fails to start with an analysis error. What is the **most likely** cause?

A. The `checkpointLocation` option is missing
B. The `withWatermark` call is missing before `groupBy`
C. The trigger is not set to `availableNow`
D. The source Delta table is empty

---

**Q2.** You need to upsert streaming data into a Delta table — existing rows must be **updated** and new rows inserted. Which approach should you use?

A. `.writeStream.format("delta").outputMode("update").start(path)` — built-in Delta sink with update mode
B. `.writeStream.format("delta").outputMode("append").start(path)` — built-in Delta sink with append mode
C. `.writeStream.foreachBatch(merge_func).outputMode("update").start()` — foreachBatch with MERGE SQL
D. `.writeStream.format("console").outputMode("complete").start()` — then manually insert from console

---

**Q3.** A streaming query crashes mid-processing. On restart with the **same** `checkpointLocation`, what happens?

A. All data is reprocessed from the beginning, producing duplicates in the sink
B. The query resumes from the last committed offset — no duplicates, no data loss
C. The query starts fresh with an empty sink, ignoring the checkpoint
D. The checkpoint is corrupted and the query refuses to start

---

**Q4.** On serverless compute, which trigger type is **supported** for Structured Streaming?

A. Default trigger (no trigger specified) — runs as fast as possible
B. `trigger(processingTime="5 seconds")` — fixed-interval micro-batches
C. `trigger(availableNow=True)` — process all available data then stop
D. `trigger(continuous="1 second")` — continuous low-latency processing

---

**Q5.** A healthcare streaming pipeline uses `withWatermark("event_time", "30 minutes")`. An event arrives with `event_time` 45 minutes behind the latest `event_time` already seen. What happens?

A. The event is processed normally and added to its window
B. The event is dropped because it is beyond the watermark threshold
C. The event is buffered in a queue for later reprocessing
D. The event causes the query to fail and restart

---

**Q6.** A healthcare device re-sends the same event multiple times. Each re-send has the same `event_id` but a **different** `event_time` (a few minutes later). You need to deduplicate on `event_id` with bounded state. Which approach is correct?

A. `.withWatermark("event_time", "20 minutes").dropDuplicates(["event_id", "event_time"])` — include event_time in the key for safety
B. `.withWatermark("event_time", "20 minutes").dropDuplicatesWithinWatermark(["event_id"])` — dedup on event_id only, bounded by watermark
C. `.dropDuplicates(["event_id"])` — dedup on event_id, no watermark needed
D. `.withWatermark("event_time", "20 minutes").dropDuplicatesWithinWatermark(["event_id", "event_time"])` — include event_time for extra safety

---

**Q7.** What error occurs if you call `dropDuplicatesWithinWatermark(["event_id"])` on a streaming DataFrame **without** first calling `withWatermark()`?

A. The query starts but silently produces wrong results (unbounded state)
B. The query fails at start with an error: "dropDuplicatesWithinWatermark is not supported without watermark"
C. The query starts but drops all rows as "too late"
D. The query starts but ignores the dedup entirely

## 9. SOLUTIONS — do not scroll here before trying

### Check Your Understanding

**Q1.** A `groupBy("status").count()` with `outputMode("append")` fails because **`.withWatermark()` is missing**. Append mode only emits rows that will never change again. For aggregations, Spark needs to know when a group/window is "final" — the watermark provides that signal. Without it, Spark would need to keep all groups in state forever.

**Q2.** A checkpoint stores three things:
1. **Offsets** — which rows from the source have already been read (prevents re-reading)
2. **State** — aggregation progress, dedup state, etc. (for stateful queries)
3. **Commit log** — which batches have been successfully written to the sink (prevents re-writing)

**Q3.** You must use **`foreachBatch`** together with a **`MERGE` SQL statement**. The built-in Delta sink only appends; it cannot update existing rows. `foreachBatch` gives you a static DataFrame per micro-batch, and you run `MERGE INTO ... WHEN MATCHED THEN UPDATE ... WHEN NOT MATCHED THEN INSERT ...` to upsert.

---

### Exam Questions

**Q1 — Answer: B**
`withWatermark` is missing before `groupBy`.
- **A** is wrong: a missing `checkpointLocation` causes a different error (`TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED`), not an analysis error about output mode.
- **B is correct**: `append` mode with aggregation requires a watermark. Without it, Spark cannot know when a window is final and refuses to start.
- **C** is wrong: the trigger type does not affect whether the query can start (it affects when it runs). On serverless, a wrong trigger causes a different error.
- **D** is wrong: an empty source table does not cause a query start failure.

**Q2 — Answer: C**
`foreachBatch` with MERGE is the correct approach.
- **A** is wrong: the built-in Delta sink does **not** support `outputMode("update")` — it only appends. Using `update` mode with the Delta sink causes an error.
- **B** is wrong: append mode only adds new rows; it cannot update existing rows.
- **C is correct**: `foreachBatch` gives you a static DataFrame per batch, and MERGE SQL performs the upsert (update existing + insert new).
- **D** is wrong: the console sink prints to stdout; you cannot use it to write to a Delta table.

**Q3 — Answer: B**
The query resumes from the last committed offset.
- **A** is wrong: the checkpoint stores offsets, so the query knows where it left off — no reprocessing.
- **B is correct**: the checkpoint's offset log, state, and commit log together enable exactly-once recovery: resume reading from the last offset, restore state, and skip already-committed batches.
- **C** is wrong: the checkpoint is used, not ignored — that's the whole point.
- **D** is wrong: a valid checkpoint from the same query is not corrupted by a crash.

**Q4 — Answer: C**
`trigger(availableNow=True)` is supported on serverless.
- **A** is wrong: the default trigger (`processingTime` with 0 interval) is not supported on serverless.
- **B** is wrong: `processingTime` is not supported on serverless (requires classic compute).
- **C is correct**: `availableNow` processes all available data in multiple batches then stops — ideal for batch-like jobs on serverless.
- **D** is wrong: `continuous` is an experimental trigger with limited operator support and is not recommended.

**Q5 — Answer: B**
The event is dropped.
- **A** is wrong: the event is 45 minutes late, which is beyond the 30-minute watermark threshold. It is dropped, not processed.
- **B is correct**: the watermark is `MAX(event_time) - 30 minutes`. An event 45 minutes behind is below the watermark and is dropped.
- **C** is wrong: Structured Streaming does not buffer late events for reprocessing — they are simply dropped.
- **D** is wrong: a late event does not cause the query to fail; it is silently dropped.

> **Note:** In practice, the watermark is a **minimum** guarantee — Spark may still process some events that arrive slightly later than the threshold. But an event 45 minutes late (15 minutes beyond a 30-minute threshold) is definitively dropped.

**Q6 — Answer: B**
`dropDuplicatesWithinWatermark(["event_id"])` with a watermark is correct.
- **A** is wrong: including `event_time` in the dedup key means re-sent events with a different timestamp are treated as new rows — duplicates are NOT caught.
- **B is correct**: `dropDuplicatesWithinWatermark` deduplicates on `event_id` only, using the watermark column (`event_time`) to bound state. Re-sent events within the watermark window are caught.
- **C** is wrong: `dropDuplicates` without a watermark produces **unbounded state** — every `event_id` is kept forever.
- **D** is wrong: including `event_time` in `dropDuplicatesWithinWatermark` has the same problem as A — re-sent events with different timestamps are treated as new rows.

**Q7 — Answer: B**
The query fails at start with an error.
- **A** is wrong: the query does not start at all — it fails during analysis.
- **B is correct**: `dropDuplicatesWithinWatermark` **requires** `withWatermark()` to be called first. Without it, Spark raises an error at query start.
- **C** is wrong: the query does not start, so no rows are dropped.
- **D** is wrong: the query does not start, so the dedup is not "ignored" — it errors.

In [0]:
%python
# ═══════════════════════════════════════════════════════════════════
# CLEANUP — Drop everything the Setup created
# Run this cell when you are done with the lesson
# ═══════════════════════════════════════════════════════════════════

spark.sql(f"DROP SCHEMA IF EXISTS {FULL_SCHEMA} CASCADE")  # Drop schema and ALL tables/volumes inside it
print(f"✅ Dropped schema {FULL_SCHEMA} and all objects within it")